<a href="https://colab.research.google.com/github/MatteoBaraldi/Machine-Learning-for-Bioengineering/blob/main/MOD-2/02_dimensionality_reductionexercise_dimensionality_reduction_solved.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practice: Dimensionality reduction algorithms

## Learning objectives
* apply PCA to a reduce the dimensionality of a high-dimensional dataset
* Select the optimal number of principal components
* repeat the exercise with IsoMap/tSNA/UMAP, and analyze how the projection depends on the model parameters

## 0. Imports and setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.manifold import Isomap
from sklearn.manifold import TSNE

np.random.seed(42)

## 1. Import and refine data

The array X includes the expression value of 2000 genes for 2700 cells obtained by single-cell RNAseq. The kind of the cell, as defined by known genetic markers, is reported in the array y.

In [ ]:
data = np.load("data_pbmc.npz")
X = data["X"]
y = data["y"]

Remove from X and y all the cells that are labelled as 'Unknown'.
* How many cells are left ?
* Create an array, y_int, with the cell types converted into integer values, and an array, labels, with the list of cell types (Suggestion: use the function np.unique)

In [ ]:
inds = y != 'Unknown'
X = X[inds,:]
y = y[inds]
labels, y_int = np.unique(y, return_inverse=True)
print('Number of samples:', X.shape[0])
print('Possible cell types:', labels)

## 2. Principal Component Analysis

* Project the data onto the first 50 principal components
* Plot the cumulative variance explained by the PCs
* Plot the data over the first 2 PCs using different markers (or colors) for the various cell kinds
* Compute the silhoutte score using the first 2 PCs and the labels in y_int. How do you interpret the silhoutte score in this case ?

In [ ]:
n_pc = 50
pca = PCA(n_components = n_pc)
pca.fit(X)
Xpca = pca.transform(X)

In [ ]:
f = plt.figure()
ax = f.add_subplot(1,1,1)
ax.plot(np.arange(n_pc), np.cumsum(pca.explained_variance_ratio_), '-ok')

In [ ]:
f = plt.figure()
ax = f.add_subplot(1,1,1)
for i_label, label in enumerate(labels):
    inds = y_int == i_label
    ax.plot(Xpca[inds,0], Xpca[inds,1], '+', label = label)
plt.legend()

In [ ]:
sc = silhouette_score(Xpca[:,:2], y_int)
print('Silhoutte score with 2 PCs:', sc)

## 3. Dimensionality reduction with IsoMap

* Use the IsoMap method to project the data over a 2-dimensional space starting from the original data X
* Change the value of the number of neighbors in the range from 5 to 20, and select the value that optimize the silhoutte score. Can you get a decent separation of the different cell types ?
* Repeat the previous points using as input the projection of the data over the first 50 principal components
* Select the number of neighbors that provides the highest silhoutte score
* Plot the data in the projected space using different colors for the various cell types

In [ ]:
ns_neighbors = np.arange(5,21,2).astype(int)
scores = np.zeros(len(ns_neighbors))
for i, n_neighbors in enumerate(ns_neighbors):
    iso = Isomap(n_components=2, n_neighbors=n_neighbors)
    Xiso = iso.fit_transform(X)
    scores[i] = silhouette_score(Xiso, y_int)
    print('Silhoutte score for IsoMap with n_neighbors {}: {}'.format(n_neighbors, scores[i]))

In [ ]:
ns_neighbors = np.arange(5,21,2).astype(int)
scores = np.zeros(len(ns_neighbors))
for i, n_neighbors in enumerate(ns_neighbors):
    iso = Isomap(n_components=2, n_neighbors=n_neighbors)
    Xiso = iso.fit_transform(Xpca)
    scores[i] = silhouette_score(Xiso, y_int)
    print('Silhoutte score for IsoMap with n_neighbors {}: {}'.format(n_neighbors, scores[i]))

In [ ]:
n_neighbor = ns_neighbors[np.argmax(scores)]
print('Using {} neighbors'.format(n_neighbor))
iso = Isomap(n_components=2, n_neighbors=n_neighbors)
Xiso = iso.fit_transform(Xpca)
f = plt.figure()
ax = f.add_subplot(1,1,1)
for i_label, label in enumerate(labels):
    inds = y_int == i_label
    ax.plot(Xiso[inds,0], Xiso[inds,1], '+', label = label)
plt.legend()

## 4. Dimensionality reduction with tSNE

* Use the TSNE method to project the data over a 2-dimensional space starting from the projection of the data over the first 50 principal components
* Test different values of perplexity and observe how the projection changes

In [ ]:
perplexity = 10
tsne = TSNE(n_components=2, perplexity=perplexity, random_state=42)
Xtsne = tsne.fit_transform(Xpca)
f = plt.figure()
ax = f.add_subplot(1,1,1)
for i_label, label in enumerate(labels):
    inds = y_int == i_label
    ax.plot(Xtsne[inds,0], Xtsne[inds,1], '+', label = label)
plt.legend()

## 4. Dimensionality reduction with UMAP

* Create a conda environment by cloning the base environment (conda create --name NAME_NEW_ENV --clone base)
* Activate the new environment (conda activate NAME_NEW_ENV)
* Install the package umap-learn from the channel conda-forge (conda install -c conda-forge umap-learn
* Import the umap library
* Use the umap method to project the data over a 2-dimensional space starting from the projection of the data over the first 50 principal components
* Test different values of the number of neighbors and observe how the projection changes

In [ ]:
import umap

In [ ]:
n_neighbors = 4
uma = umap.UMAP(n_components=2, n_neighbors=n_neighbors, min_dist=min_dist, random_state=42, n_jobs = 1)
Xuma = uma.fit_transform(Xpca)
f = plt.figure()
ax = f.add_subplot(1,1,1)
for i_label, label in enumerate(labels):
    inds = y_int == i_label
    ax.plot(Xuma[inds,0], Xuma[inds,1], '+', label = label)
plt.legend()

## 5. Clustering in the low dimensional space

* Clusterize the data in the low-dimensional space obtained by UMAP with DBSCAN. Optimize the parameters of DBSCAN by looking at the clusterized data in the low-dimensional space
* Use the function imshow to plot an heatmap of gene expression values. Restrict the scale of the heatmap in the range from -0.1 to 0.1
* Order the cells based on the DBSCAN clustering results

In [ ]:
from sklearn.cluster import DBSCAN
model = DBSCAN(eps=0.5, min_samples=5).fit(Xuma)
f = plt.figure()
ax = f.add_subplot(1,1,1)
for i_label in range(n_clusters):
    inds = model.labels_ == i_label
    ax.plot(Xuma[inds,0], Xuma[inds,1], '+', label = i_label)
plt.legend()
inds = model.labels_ == -1
ax.plot(Xuma[inds,0], Xuma[inds,1], 'xk', label = -1)

In [ ]:
inds = np.argsort(model.labels_)
plt.imshow(X[inds], cmap="viridis", vmin = -0.1, vmax = 0.1)
plt.colorbar()
plt.show()